**Alunas:**
Pamella Lissa Sato Tamura (2254107) e Raquel de Oliveira (2399113).

# Etapa 2 - Limpeza e Integração dos Dados

### Notas:

**Lembretes - Dataset Extra:**

Quando utilizar múltiplas fontes, siga estas etapas:
  1. Padronizar nomenclaturas e tipos de dados entre fontes;
  2. Resolver conflitos quando fontes divergem para o mesmo registro;
  3. Detectar e tratar registros duplicados;
  4. Criar identificadores únicos consistentes.

**Limpeza de Dados**

Implemente soluções para os principais problemas de qualidade:

  1. **Missing values:** escolher estratégias de imputação adequadas;
  2. **Outliers:** detectar valores extremos e decidir sobre tratamento;
  3. **Inconsistências:** verificar dependências entre variáveis (dependências funcionais, se existirem) e corrigir possíveis erros;
  4. **Padronização:** uniformizar formatos de datas, textos e códigos.

## Imports

In [8]:
import pandas as pd

In [10]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Carregando os Datasets

In [27]:
raw_exports = pd.read_csv("../data/exportacao_2025.csv", sep=";")
# Encoding
raw_ncm = pd.read_csv("../data/codigo_ncm.csv", sep=";", encoding="latin1")
raw_countries = pd.read_csv("../data/codigo_pais.csv", sep=";", encoding="latin1")

columns_mapping = {
    "CO_ANO": "year",
    "CO_MES": "month",
    "CO_NCM": "ncm_code",
    "SG_UF_NCM": "origin_state",
    "CO_VIA": "transport_route",
    "KG_LIQUIDO": "net_weight_kg",
    "VL_FOB": "fob_value_usd",
    "CO_URF": "urf_code",
    "CO_PAIS": "country_code",
    "CO_UNID": "statistical_quantity",
    "QT_ESTAT": "statistical_unit_code",
    "NO_NCM_POR": "ncm_name_ptbr",
    "NO_PAIS": "country_name"
}

export_df = raw_exports.rename(columns=columns_mapping)
ncm_df = raw_ncm.rename(columns=columns_mapping)
countries_df = raw_countries.rename(columns=columns_mapping)

## Inspeção Inicial

Os comentários dos outputs foram feitos antes das modificações feitas nos datasets nas etapas anteriores.

### Dataset - Exportações

In [13]:
export_df.head()

,year,month,ncm_code,statistical_quantity,country_code,origin_state,transport_route,urf_code,statistical_unit_code,net_weight_kg,fob_value_usd
0,2025,8,71039900,19,687,RS,4,817700,61690,13,203
1,2025,12,2071412,10,741,SC,1,817800,83835,83835,111324
2,2025,11,33059000,10,767,SP,4,817600,792,753,22012
3,2025,11,56031330,10,845,SP,7,1017701,2437,2437,8458
4,2025,6,21069090,10,365,SP,1,817800,4800,4800,17304


A maior parte dos atributos foi carregada como inteiro, enquanto a UF foi identificada como variável categórica. Apesar disso, vários códigos numéricos representam categorias e posteriormente serão convertidos para string.

In [14]:
export_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1709746 entries, 0 to 1709745
Data columns (total 11 columns):
 #   Column                 Dtype
---  ------                 -----
 0   year                   int64
 1   month                  int64
 2   ncm_code               int64
 3   statistical_quantity   int64
 4   country_code           int64
 5   origin_state           str  
 6   transport_route        int64
 7   urf_code               int64
 8   statistical_unit_code  int64
 9   net_weight_kg          int64
 10  fob_value_usd          int64
dtypes: int64(10), str(1)
memory usage: 143.5 MB


O dataset principal possui 1.709.746 registros e 11 atributos relacionados às exportações brasileiras de 2025 (valores antes das modificações feitas a seguir).

In [15]:
export_df.describe()

,year,month,ncm_code,statistical_quantity,country_code,transport_route,urf_code,statistical_unit_code,net_weight_kg,fob_value_usd
count,1709746.0,1.709746e+06,1.709746e+06,1.709746e+06,1.709746e+06,1.709746e+06,1.709746e+06,1.709746e+06,1.709746e+06,1.709746e+06
mean,2025.0,6.680403e+00,4.940779e+07,1.065404e+01,3.873270e+02,2.999435e+00,8.003287e+05,3.524104e+05,5.028979e+05,2.037019e+05
std,0.0,3.417686e+00,3.208940e+07,1.711613e+00,2.372586e+02,2.633118e+00,3.609370e+05,3.778164e+07,3.858018e+07,4.784208e+06
min,2025.0,1.000000e+00,1.012100e+06,1.000000e+01,1.300000e+01,0.000000e+00,1.176000e+05,0.000000e+00,0.000000e+00,0.000000e+00
25%,2025.0,4.000000e+00,1.905320e+07,1.000000e+01,1.690000e+02,1.000000e+00,8.176000e+05,5.000000e+00,5.000000e+00,8.800000e+01
50%,2025.0,7.000000e+00,4.504900e+07,1.000000e+01,3.720000e+02,1.000000e+00,8.178000e+05,3.200000e+01,3.900000e+01,7.650000e+02
75%,2025.0,1.000000e+01,8.424899e+07,1.100000e+01,5.860000e+02,4.000000e+00,9.178000e+05,5.210000e+02,8.850000e+02,1.155000e+04
max,2025.0,1.200000e+01,9.706900e+07,2.200000e+01,8.900000e+02,1.500000e+01,9.999999e+06,1.369865e+10,1.369865e+10,1.009546e+09


As variáveis numéricas apresentam forte assimetria, com diferenças grandes entre mediana e valor máximo. Isso indica presença de exportações extremamente volumosas ou valiosas, algo esperado em dados de comércio exterior. Também foram identificados valores mínimos iguais a zero em atributos como KG_LIQUIDO e VL_FOB, que vão ser investigados posteriormente.

In [16]:
export_df.isnull().sum()

year                     0
month                    0
ncm_code                 0
statistical_quantity     0
country_code             0
origin_state             0
transport_route          0
urf_code                 0
statistical_unit_code    0
net_weight_kg            0
fob_value_usd            0
dtype: int64

O dataset principal não apresentou valores nulos em nenhuma coluna.

### Dataset - NCM (Produtos)

O dataset NCM contém informações complementares sobre classificação e descrição dos produtos exportados. Os registros mostram descrições em português, espanhol e inglês, além de classificações econômicas e fiscais.

In [29]:
# Lista de colunas que não trazem poder preditivo para a via de transporte
columns_to_drop = [
    "CO_SH6",
    "CO_PPE",
    "CO_PPI",
    "CO_FAT_AGREG",
    "CO_CUCI_ITEM",
    "CO_CGCE_N3",
    "CO_SIIT",
    "CO_ISIC_CLASSE",
    "CO_EXP_SUBSET",
    "NO_NCM_ESP",
    "NO_NCM_ING"
]

ncm_df = ncm_df.drop(columns=columns_to_drop, errors="ignore")

In [30]:
ncm_df.head()

,ncm_code,statistical_quantity,ncm_name_ptbr
0,90183930,11,Lancetas para vacinação e cautérios
1,90183990,11,"Outros instrumentos semelhantes a seringas, agulhas, catéteres, etc"
2,90183991,11,"Artigo para fístula arteriovenosa, composto de agulha, base de fixação tipo borboleta, tubo plástico com conector e obturador"
3,90183999,11,"Outros instrumentos semelhantes a seringas, agulhas, catéteres, etc"
4,90184100,11,"Aparelhos dentários de brocar, mesmo combinados numa base comum com outros equipamentos dentários"


O dataset NCM possui 13.744 registros e 3 colunas (após o drop).

In [31]:
ncm_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13744 entries, 0 to 13743
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   ncm_code              13744 non-null  int64
 1   statistical_quantity  13744 non-null  int64
 2   ncm_name_ptbr         13744 non-null  str  
dtypes: int64(2), str(1)
memory usage: 322.3 KB


In [20]:
ncm_df.describe()

,ncm_code,statistical_quantity
count,1.374400e+04,13744.000000
mean,5.059577e+07,10.503492
std,2.794693e+07,1.298717
min,1.011010e+06,10.000000
25%,2.921573e+07,10.000000
50%,4.206500e+07,10.000000
75%,8.420101e+07,11.000000
max,9.999996e+07,22.000000


In [21]:
ncm_df.isnull().sum()

ncm_code                0
statistical_quantity    0
NO_NCM_POR              0
dtype: int64

### Dataset -Países

In [32]:
columns_to_drop = [
    "CO_PAIS_ISON3",
    "CO_PAIS_ISOA3",
    "NO_PAIS_ING",
    "NO_PAIS_ESP"
]

countries_df = countries_df.drop(columns=columns_to_drop, errors="ignore")

In [33]:
countries_df.head()

,country_code,country_name
0,0,Não Definido
1,13,Afeganistão
2,15,"Aland, Ilhas"
3,17,Albânia
4,20,"Alboran-Perejil, Ilhas"


O dataset de países (`codigo_pais`) possui 281 registros e não apresenta valores ausentes.

In [24]:
countries_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 281 entries, 0 to 280
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   country_code  281 non-null    int64
 1   NO_PAIS       281 non-null    str  
dtypes: int64(1), str(1)
memory usage: 4.5 KB


In [25]:
countries_df.describe()

,country_code
count,281.000000
mean,449.469751
std,266.870822
min,0.000000
25%,232.000000
50%,447.000000
75%,690.000000
max,999.000000


In [34]:
countries_df.isnull().sum()

country_code    0
country_name    0
dtype: int64

## Conversão de Tipos

As colunas foram convertidas para string, pois representam categorias e identificadores, não valores numéricos para cálculo. A conversão também evita problemas durante a integração entre os datasets.

In [35]:
# Convertendo colunas categóricas para string
cols_export_str = [
    "year",
    "month",
    "ncm_code",
    "statistical_quantity",
    "country_code",
    "transport_route",
    "urf_code"
]

export_df[cols_export_str] = export_df[cols_export_str].astype(str)
ncm_df[["ncm_code", "statistical_quantity"]] = ncm_df[["ncm_code", "statistical_quantity"]].astype(str)
countries_df["country_code"] = countries_df["country_code"].astype(str)

Colunas que representam identificadores e categorias convertidas para string, preservando a semântica dos dados e garantindo consistência na integração.